In [ ]:
import os
import json
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_core.documents import Document

LLM_MODEL_NAME="jaro-jm/llm:latest"
os.environ["HSA_OVERRIDE_GFX_VERSION"]="11.0.0"

llm_text = ChatOllama(model=LLM_MODEL_NAME, temperature=0, reasoning=False, num_ctx=35000, num_predict=35000)

text = """Donald Trump od początku traktował konflikt w Ukrainie jako nieswoją wojnę. Nie miał cierpliwości ani zrozumienia dla ukraińskich racji. Błędnie zakładał też, że zdoła oderwać Rosję od sojuszu z Chinami. Tymczasem oba kraje mocno zacieśniły współpracę gospodarczą, a Chiny dostarczają dziś Rosjanom maszyny i technologie, na które monopol miał kiedyś Zachód. Moskwa i Pekin pamiętają czasy, gdy były podzielone. Putin i Xi są zgodni, że rozpad Związku Radzieckiego nie był koniecznością, tylko skutkiem błędów Kremla. Rosję i Chiny dzieli przepaść kulturalna, bo Rosjanie nie chcą być Chińczykami, ale to akurat ułatwia relacje. Współpraca z Europą grozi w oczach Putina zarażeniem rosyjskiego społeczeństwa „demokratycznym wirusem”, a w kontaktach z Chinami takiego ryzyka nie ma. Rosja może jednak wpaść w nieodwracalną zależność od Chin. Decydując się na inwazję na Ukrainę, Putin chciał przede wszystkim zasilić ludność Rosji dziesiątkami milionów Słowian, co było dla niego ważniejsze od zdobyczy terytorialnych. W tle są obawy przed chińską dominacją."""

In [ ]:
extraction_prompt = ChatPromptTemplate.from_messages([
    ("system", """You analyze text for the purpose of building a knowledge graph.
Your task is to extract ONLY 12-17 most relevant:
 - entities,
 - events,
 - concepts,
 that can function as nodes in the knowledge graph.

Specifically identify:
- persons (individual human actors),
- organizations (companies, institutions, movements),
- locations (geographical and administrative places),
- events (historical, social, political, technical, scientific, etc.),
- time expressions (dates, periods, time ranges),
- key thematic concepts (ideas, technologies, phenomena, theories).

**Rules:**
- Return only semantically meaningful elements, that can exist independently as nodes in a knowledge graph.
- Use canonical names where possible (e.g. “European Union” instead of abbreviations, if context allows).
- Do not duplicate items.
- Do not add category labels, keys, explanations or metadata.
- Return everything in Polish.
- Avoid returning multiple surface forms of the same entity or concept.
  If different names refer to the same real-world entity (e.g. „ChRL”, „Chiny”, „Chińska Republika Ludowa”),
  return only one canonical form, chosen based on common usage in context.

**Output format:**
- A single line of text.
- All extracted elements separated by commas.
- No JSON, no lists, no bullet points, no headings, no comments."""),
    ("user", """Text:
    Donald Trump nie ogranicza się do przebudowy Białego Domu. Zmienia też to, jak Amerykanie patrzą
na świat. Wenezuela liczy się dla niego bardziej niż Ukraina, bo to z Ameryki Łacińskiej
pochodzą największe kłopoty dzisiejszych Stanów: nielegalna imigracja i przemyt narkotyków.
Ważny jest również Pacyfik i rywalizacja z Chinami o hegemonię w tym regionie.
Trump uważa, że USA traciły na porządku, który powstał po zakończeniu zimnej wojny.
Wciąż trwa spór o to, czy przyjęcie ChRL do WTO było dobrą decyzją.
Jednocześnie Ameryka wiele zyskiwała, między innymi dzięki temu,
że dolar pozostawał walutą rezerwową świata."""),
    ("ai", """Donald Trump, Biały Dom, Stany Zjednoczone, Amerykanie, Wenezuela, Ukraina, Ameryka Łacińska,
nielegalna imigracja, przemyt narkotyków, Pacyfik, Chiny, hegemonia, zimna wojna,
Światowa Organizacja Handlu, dolar, waluta rezerwowa świata"""),
    ("user", "Text: {text}")
])

extraction_chain = extraction_prompt | llm_text
extracted_content_raw = extraction_chain.invoke({"text": text})
extracted_content = extracted_content_raw.content
print(extracted_content_raw)

In [ ]:
nodes = [
    item.strip()
    for item in extracted_content.split(",")
    if item.strip()
]

seen = set()
nodes = [x for x in nodes if not (x in seen or seen.add(x))]
print(nodes)

In [ ]:
qa_generator_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an expert assistant for generating high-quality question–answer pairs from text.

Rules:
- Generate 10-12 question–answer pairs in Polish.
- Questions and answers MUST cover ONLY the most important aspects of the text.
- Do NOT ask generic or meta-level questions about the text itself. Forbidden example of a question: "Jakie jest główne zagadnienie omawiane w tekście?"
- Questions must test understanding of the content, not summarize it at a high level.
- Avoid trivial, vague, or purely definitional questions.
- Answers must be concise, precise, and based only on the provided text.

Output format:
- Return ONLY valid JSON.
- The JSON must be a list of objects.
- Each object must have exactly two fields: "question" and "answer".
- Do NOT include markdown, comments, or any text outside the JSON.

Example output:
[
  {{
    "question": "Jaką rolę pełni mechanizm X w opisywanym procesie?",
    "answer": "Mechanizm X odpowiada za ..."
  }},
  {{
    "question": "Jakie konsekwencje według tekstu wynikają z zastosowania rozwiązania Y?",
    "answer": "Zastosowanie rozwiązania Y prowadzi do ..."
  }}
]"""),
    ("user", """Generate question–answer pairs from the following text.
Text:\n{text}""")
])

llm_chain = qa_generator_prompt | llm_text
output = llm_chain.invoke({"text": text}).content

qa_pairs = json.loads(output)
all_pairs = [f"Pytanie {i+1}: {pair['question']}\nOdpowiedź {i+1}: {pair['answer']}" for i, pair in enumerate(qa_pairs)]
single_sting = "\n" + "-" * 40 + "\n"
single_sting = "\n\n".join(all_pairs)
print(single_sting)

In [ ]:
relations_prompt = ChatPromptTemplate.from_messages([
    ("system", """You extract factual relations for building a knowledge graph.

You are given:
- List of nodes,
- Question–answer pairs in Polish.

Your task is to identify 12–15 explicitly stated relations between the provided nodes, using the Q&A pairs as context.

Rules:
- Only use nodes from the provided list.
- Each triple must have a subject, a concise relation, and an object.
- Relations must be precise and semantically meaningful. Avoid too short relations.
- Do not omit the object. If no valid object exists, do not create a triple.
- Multiple relations between the same nodes should be returned as separate triples.
- Direction matters: subject -> relation -> object.
- Focus only on facts explicitly stated in the Q&A pairs, not opinions or interpretations.

Output format:
- JSON only.
- Must strictly match: {{"triples":[{{"subject":"","relation":"","object":""}}]}}
- No explanations or comments.
- If no valid relations exist, return: {{"triples":[]}}"""),
    ("user", "Q&A pairs:\n{pairs}\n\nNodes:\n{nodes}")
])

relations_chain = relations_prompt | llm_text
relations_output = relations_chain.invoke({
    "pairs": single_sting,
    "nodes": ", ".join(nodes)
})

In [ ]:
relations_output_json = json.loads(relations_output.content)
print("Relations:")
for triple in relations_output_json['triples']:
    print(f"{triple['subject']} -> {triple['relation']} -> {triple['object']}")

In [ ]:
llm_transformer = LLMGraphTransformer(
    llm=llm_text,
    allowed_nodes=nodes,
    node_properties=True,
    relationship_properties=True,
    strict_mode=False,
    additional_instructions="""Extract ONLY the most important Nodes (Entities, Concepts, Events) and Relationships between them in POLISH. A triplet (Node A -> Relation -> Node B) must be a complete, factual sentence on its own, understandable without reading the original text.""",
)

graph_documents = llm_transformer.convert_to_graph_documents([Document(single_sting)])

for doc_idx, graph_doc in enumerate(graph_documents, start=1):
    print(f"\n=== DOCUMENT {doc_idx} ===")
    print("NODES:")
    for i, node in enumerate(graph_doc.nodes, start=1):
        print(f"  {i:02d}. {node.id} ({node.type})")

    print("RELATIONSHIPS:")
    for i, rel in enumerate(graph_doc.relationships, start=1):
        source = rel.source.id
        target = rel.target.id
        rel_type = rel.type
        print(f"  {i:02d}. {source} ──[{rel_type}]──▶ {target}")

In [ ]:
llm_transformer = LLMGraphTransformer(
    llm=llm_text,
    allowed_nodes=nodes,
    node_properties=True,
    relationship_properties=True,
    strict_mode=False,
    additional_instructions="""Extract ONLY the most important Nodes (Entities, Concepts, Events) and Relationships between them in POLISH. A triplet (Node A -> Relation -> Node B) must be a complete, factual sentence on its own, understandable without reading the original text. Avoid single-word relationships where possible.""",
)

graph_documents = llm_transformer.convert_to_graph_documents([Document(single_sting)])

for doc_idx, graph_doc in enumerate(graph_documents, start=1):
    print(f"\n=== DOCUMENT {doc_idx} ===")
    print("NODES:")
    for i, node in enumerate(graph_doc.nodes, start=1):
        print(f"  {i:02d}. {node.id} ({node.type})")

    print("RELATIONSHIPS:")
    for i, rel in enumerate(graph_doc.relationships, start=1):
        source = rel.source.id
        target = rel.target.id
        rel_type = rel.type
        print(f"  {i:02d}. {source} ──[{rel_type}]──▶ {target}")

In [ ]:
llm_transformer = LLMGraphTransformer(
    llm=llm_text,
    allowed_nodes=nodes,
    node_properties=True,
    relationship_properties=True,
    strict_mode=True,
    additional_instructions="""Extract ONLY the most important Nodes (Entities, Concepts, Events) and Relationships between them in POLISH. A triplet (Node A -> Relation -> Node B) must be a complete, factual sentence on its own, understandable without reading the original text. Avoid single-word relationships where possible.""",
)

graph_documents = llm_transformer.convert_to_graph_documents([Document(single_sting)])

for doc_idx, graph_doc in enumerate(graph_documents, start=1):
    print(f"\n=== DOCUMENT {doc_idx} ===")
    print("NODES:")
    for i, node in enumerate(graph_doc.nodes, start=1):
        print(f"  {i:02d}. {node.id} ({node.type})")

    print("RELATIONSHIPS:")
    for i, rel in enumerate(graph_doc.relationships, start=1):
        source = rel.source.id
        target = rel.target.id
        rel_type = rel.type
        print(f"  {i:02d}. {source} ──[{rel_type}]──▶ {target}")

In [ ]:
single_sting = """Pytanie 1: Jak Trump postrzegał konflikt w Ukrainie?
Odpowiedź 1: Trump uważał go za nieswoją wojnę, brakowało mu cierpliwości i zrozumienia dla ukraińskich racji.

Pytanie 2: Jakie błędne założenie miał Trump dotyczące Rosji i Chin?
Odpowiedź 2: Trump uważał, że może oderwać Rosję od sojuszu z Chinami.

Pytanie 3: Jakie elementy współpracy gospodarczej łączą Rosję i Chiny?
Odpowiedź 3: Chiny są w stanie dostarczyć Rosjanom maszyny i technologie, które kiedyś monopol miał Zachód.

Pytanie 4: Co Moskwa i Pekin wyciągnęły z czasów podziału Związku Radzieckiego?
Odpowiedź 4: Obie strony uważają, że rozpad nie był koniecznością, lecz wynikiem błędów Kremla.

Pytanie 5: Jaka jest główna przeszkoda kulturowa między Rosją a Chinami?
Odpowiedź 5: Przepaść kulturalna, brak Rosjan, którzy chcieliby być Chińczykami.

Pytanie 6: Dlaczego Putin uważa współpracę z Europą za ryzykowną?
Odpowiedź 6: Ryzyko zarażenia się przez rosyjskie społeczeństwo „demokratycznym wirusem”.

Pytanie 7: Jakie ryzyko widzi Putin w zależności od Chin?
Odpowiedź 7: Rosja może wpaść w nieodwracalną zależność od Chin.

Pytanie 8: Jaki był główny motyw Putina przy inwazji na Ukrainę?
Odpowiedź 8: Zasilić ludność Rosji dziesiątkami milionów Słowian, ważniejsze niż terytorialne zdobycze.

Pytanie 9: Jakie obawy są w tle decyzji Putina?
Odpowiedź 9: Obawy przed chińską dominacją.

Pytanie 10: Dlaczego współpraca z Chinami nie niesie ryzyka „demokratycznego wirusa”?
Odpowiedź 10: Brak takiego ryzyka w relacjach z Chinami, w przeciwieństwie do Europy.

Pytanie 11: Co oznacza „demokratyczny wirus” w kontekście Putina?
Odpowiedź 11: Zarażenie społeczeństwa rosyjskiego ideami demokratycznymi.

Pytanie 12: Jakie konsekwencje może mieć nieodwracalna zależność Rosji od Chin?
Odpowiedź 12: Ograniczenie suwerenności i utrata niezależności w decyzjach gospodarczych i politycznych."""

nodes = ['Donald Trump', 'Vladimir Putin', 'Xi Jinping', 'Rosja', 'Chiny', 'Ukraina', 'Związek Radziecki', 'Moskwa', 'Pekin', 'Europa', 'konflikt w Ukrainie', 'inwazja na Ukrainę', 'rozpad Związku Radzieckiego', 'przepaść kulturalna', 'nieodwracalna zależność od Chin']


relations_prompt = ChatPromptTemplate.from_messages([
    ("system", """You extract factual relations for building a knowledge graph. You are given:
- List of nodes,
- Question–answer pairs in Polish.

Rules:
- Extract ONLY the most important triplets in POLISH.
- A triplet (Node A -> Relation -> Node B) must be a complete, factual sentence on its own, understandable without reading the original text.
- Avoid single-word relationships where possible. Perfect relationship should be in between 1 to 3 words and provide whole context without a need to look at the source text.
- Only use nodes from the provided list.
- Each triple must have a subject, a concise relation, and an object.
- Do not omit the object. If no valid object exists, do not create a triple.
- Multiple relationships between the same nodes should be returned as separate triples.
- Direction matters: subject -> relation -> object.
- Focus only on facts explicitly stated in the Q&A pairs, not opinions or interpretations.

Output format:
- JSON only.
- Must strictly match: {{"triples":[{{"subject":"","relation":"","object":""}}]}}
- No explanations or comments.
- If no valid relations exist, return: {{"triples":[]}}"""),
    ("user", "Q&A pairs:\n{pairs}\n\ nNodes:\n{nodes}")
])

relations_chain = relations_prompt | llm_text
relations_output = relations_chain.invoke({
    "pairs": single_sting,
    "nodes": ", ".join(nodes)
})
relations_output_json = json.loads(relations_output.content)
print("Relations:")
for triple in relations_output_json['triples']:
    print(f"{triple['subject']} -> {triple['relation']} -> {triple['object']}")

In [ ]:
single_sting = """Pytanie 1: Jak Trump postrzegał konflikt w Ukrainie?
Odpowiedź 1: Trump uważał go za nieswoją wojnę, brakowało mu cierpliwości i zrozumienia dla ukraińskich racji.

Pytanie 2: Jakie błędne założenie miał Trump dotyczące Rosji i Chin?
Odpowiedź 2: Trump uważał, że może oderwać Rosję od sojuszu z Chinami.

Pytanie 3: Jakie elementy współpracy gospodarczej łączą Rosję i Chiny?
Odpowiedź 3: Chiny są w stanie dostarczyć Rosjanom maszyny i technologie, które kiedyś monopol miał Zachód.

Pytanie 4: Co Moskwa i Pekin wyciągnęły z czasów podziału Związku Radzieckiego?
Odpowiedź 4: Obie strony uważają, że rozpad nie był koniecznością, lecz wynikiem błędów Kremla.

Pytanie 5: Jaka jest główna przeszkoda kulturowa między Rosją a Chinami?
Odpowiedź 5: Przepaść kulturalna, brak Rosjan, którzy chcieliby być Chińczykami.

Pytanie 6: Dlaczego Putin uważa współpracę z Europą za ryzykowną?
Odpowiedź 6: Ryzyko zarażenia się przez rosyjskie społeczeństwo „demokratycznym wirusem”.

Pytanie 7: Jakie ryzyko widzi Putin w zależności od Chin?
Odpowiedź 7: Rosja może wpaść w nieodwracalną zależność od Chin.

Pytanie 8: Jaki był główny motyw Putina przy inwazji na Ukrainę?
Odpowiedź 8: Zasilić ludność Rosji dziesiątkami milionów Słowian, ważniejsze niż terytorialne zdobycze.

Pytanie 9: Jakie obawy są w tle decyzji Putina?
Odpowiedź 9: Obawy przed chińską dominacją.

Pytanie 10: Dlaczego współpraca z Chinami nie niesie ryzyka „demokratycznego wirusa”?
Odpowiedź 10: Brak takiego ryzyka w relacjach z Chinami, w przeciwieństwie do Europy.

Pytanie 11: Co oznacza „demokratyczny wirus” w kontekście Putina?
Odpowiedź 11: Zarażenie społeczeństwa rosyjskiego ideami demokratycznymi.

Pytanie 12: Jakie konsekwencje może mieć nieodwracalna zależność Rosji od Chin?
Odpowiedź 12: Ograniczenie suwerenności i utrata niezależności w decyzjach gospodarczych i politycznych."""

nodes = ['Donald Trump', 'Vladimir Putin', 'Xi Jinping', 'Rosja', 'Chiny', 'Ukraina', 'Związek Radziecki', 'Moskwa', 'Pekin', 'Europa', 'konflikt w Ukrainie', 'inwazja na Ukrainę', 'rozpad Związku Radzieckiego', 'przepaść kulturalna', 'nieodwracalna zależność od Chin']


relations_prompt = ChatPromptTemplate.from_messages([
    ("system", """You extract factual relations for building a knowledge graph.

You are given:
- List of nodes.
- Question–answer pairs in Polish.

Rules:
- Extract ONLY the most important triplets in POLISH.
- Only use nodes from the provided list.
- Relations must be short, reusable semantic types (1–3 words).
- Each triple must have a subject, a concise relation, and an object.
- Do not omit the object. If no valid object exists, do not create a triple.
- Multiple relationships between the same nodes should be returned as separate triples.
- Direction matters: subject -> relation -> object.
- When necessary, split a complex relationship into separate triples.
  In such cases:
  1. Extract an epistemic relation
  2. Extract factual or causal relations separately

Output format rules:
- JSON only.
- Must strictly match: {{"triples":[{{"subject":"","relation":"","object":""}}]}}
- No explanations or comments.
- If no valid relations exist, return: {{"triples":[]}}"""),
    ("user", "Extract 10-15 triplets for question–answer pairs and nodes provided below.\nQuestion–answer pairs:\n{pairs}\n\nList of nodes:\n{nodes}")
])

relations_chain = relations_prompt | llm_text
relations_output = relations_chain.invoke({
    "pairs": single_sting,
    "nodes": ", ".join(nodes)
})
relations_output_json = json.loads(relations_output.content)
print("Relations:")
for triple in relations_output_json['triples']:
    print(f"{triple['subject']} -> {triple['relation']} -> {triple['object']}")